In [1]:
import pandas as pd
import numpy as np

import sys
sys.path.append("..")
from Src.data_modules import *

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report,cohen_kappa_score
from sklearn.preprocessing import StandardScaler

# Data loading

In [2]:
data_model = pd.read_csv("data_model.csv", sep=";", index_col=0)

y = data_model["target"]
X = data_model.drop(columns=["target"])

In [3]:
test_data_model = pd.read_csv("test_data_model.csv", sep=";", index_col=0)

X_test = data_model.drop(columns=["target"])

In [4]:
data_model

,mode_0,mode_1,mode_2,mode_3,amplitude,target
0,2.108017,3.766613,0.000000,2.584784,20056.871,0
1,5.632278,5.586515,0.702054,2.434528,18176.380,0
2,1.129253,4.057986,0.000000,1.383936,18218.500,0
3,1.247747,3.728297,0.009119,2.008376,18827.771,0
4,2.367924,3.358915,0.000000,1.701230,19442.460,0
...,...,...,...,...,...,...
261750,14.223799,6.338595,1.862412,1.812356,71428.766,0
261751,24.995213,14.817390,0.599471,2.269399,79437.570,0
261752,10.944218,6.434064,3.183469,2.152396,88409.680,0
261753,11.556972,6.026230,2.642583,1.532516,88166.570,0


In [10]:
test_data_model

,mode_0,mode_1,mode_2,mode_3,amplitude
0,21.219572,9.470988,7.524747,1.455186,1.541605e+06
1,17.731410,8.321794,9.744177,1.414656,2.239955e+06
2,12.736122,7.476161,3.518530,1.574906,2.188355e+06
3,13.011054,4.569553,2.304483,0.869931,2.182809e+06
4,11.482360,7.111195,2.449824,0.011326,2.186340e+06
...,...,...,...,...,...
112610,7.261078,3.395464,1.797075,0.280240,2.140521e+03
112611,14.427754,6.376949,6.853260,1.828387,3.069335e+03
112612,23.947513,11.630581,5.590073,0.000000,-6.372085e+03
112613,6.617773,3.476538,3.829220,1.788417,-7.140743e+03


# Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X, y)

LogisticRegression(max_iter=1000)

In [ ]:
# Normaliser les données d'entrée
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
# Diviser les données en ensembles d'entraînement et de test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Créer et entraîner le modèle de régression logistique
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Prédire les résultats sur les données de test
y_pred = model.predict(X_test)

# Évaluer les performances du modèle
accuracy = accuracy_score(y_test, y_pred)
print(f"Exactitude du modèle : {accuracy:.2f}")

# Afficher un rapport détaillé sur les performances
print("Rapport de classification :")
print(classification_report(y_test, y_pred))

# Calcul du coefficient de Kappa
kappa_score = cohen_kappa_score(y_test, y_pred)

print(f"Score Kappa : {kappa_score}")

c:\Users\planc\anaconda3\envs\env_EEG_project_ML\Lib\site-packages\scipy\optimize\_linesearch.py:312: LineSearchWarning: The line search algorithm did not converge
  alpha_star, phi_star, old_fval, derphi_star = scalar_search_wolfe2(
c:\Users\planc\anaconda3\envs\env_EEG_project_ML\Lib\site-packages\sklearn\utils\optimize.py:100: LineSearchWarning: The line search algorithm did not converge
  ret = line_search_wolfe2(


Exactitude du modèle : 0.92
Rapport de classification :
              precision    recall  f1-score   support

           0       0.91      0.90      0.90     31485
           1       0.93      0.94      0.94     47042

    accuracy                           0.92     78527
   macro avg       0.92      0.92      0.92     78527
weighted avg       0.92      0.92      0.92     78527

Score Kappa : 0.8412879162753432


In [22]:
def format_array_to_target_format(array, record_number, nb_points):

    formatted_target = []
    for i in range(5):
        channel_encoding = (i + 1) * 100000
        record_number_encoding = record_number * 1000000
        for j in range(nb_points):
            formatted_target.append(
                {
                    "identifier": record_number_encoding + channel_encoding + j,
                    "target": array[i][j],
                }
            )
    return formatted_target

In [24]:
results = []

# Set 4
X_test_4 = test_data_model[:66020]
preds = model.predict(X_test_4)
sublists = [preds[i:i + 13204] for i in range(0, len(preds), 13204)]

formatted_preds = format_array_to_target_format(sublists, 4, 13204)
results.extend(formatted_preds)

# Set 5
X_test_5 = test_data_model[66020:]
preds = model.predict(X_test_5)
sublists = [preds[i:i + 9319] for i in range(0, len(preds), 9319)]

formatted_preds = format_array_to_target_format(sublists, 5, 9319)
results.extend(formatted_preds)

df = pd.DataFrame(results)
df.to_csv("../Results/my_submission.csv",index = False)